# DUT capture throughput (10k traces)

Prerequisite: CW305 + VCU programmed, PLLs/DCO up, and an eth RX thread already running
(from `dut_power_trace_eth.ipynb`). This cell measures sustained encrypt + UDP drain rate.


Run the bring-up + eth RX cells in `dut_power_trace_eth.ipynb` first (same kernel), then run the cell below.


In [ ]:
N_TP = 10_000
QUEUE_SLOTS = 16
BACKPRESSURE = QUEUE_SLOTS - 4

cipher = AES.new(bytes(KEY), AES.MODE_ECB)
expected = list(cipher.encrypt(bytes(PLAINTEXT)))
key_w = bytearray(KEY[::-1])
pt_w = bytearray(PLAINTEXT[::-1])

with eth_lock:
    eth_traces.clear()
    eth_rx = 0

print(f"Starting throughput run: {N_TP} traces")
t_encrypt0 = time.time()
ok = 0
stuck = 0

for i in range(N_TP):
    while True:
        with eth_lock:
            nrx = eth_rx
        if (i - nrx) < BACKPRESSURE:
            break
        time.sleep(0.0005)

    target.fpga_write(target.REG_CRYPT_KEY, key_w)
    target.fpga_write(target.REG_CRYPT_TEXTIN, pt_w)
    target.fpga_write(target.REG_CRYPT_GO, [1])

    deadline = time.time() + 5.0
    busy = 1
    while time.time() < deadline:
        busy = target.fpga_read(target.REG_CRYPT_GO, 1)[0]
        if busy == 0:
            break
        time.sleep(0.0002)
    if busy != 0:
        stuck += 1
        target.fpga_write(REG_HS_RST, [1])
        time.sleep(0.05)
        target.fpga_write(target.REG_CRYPT_GO, [1])
        deadline = time.time() + 5.0
        while time.time() < deadline:
            busy = target.fpga_read(target.REG_CRYPT_GO, 1)[0]
            if busy == 0:
                break
            time.sleep(0.0002)
        if busy != 0:
            raise TimeoutError(f"throughput stuck at i={i}, eth_rx={eth_rx}, stuck={stuck}")

    if (i & 0x3FF) == 0:
        ct = bytearray(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))[::-1]
        assert list(ct) == expected, f"ct mismatch at i={i}"

    ok += 1
    if (i + 1) % 500 == 0 or i == 0:
        elapsed = time.time() - t_encrypt0
        with eth_lock:
            nrx = eth_rx
        print(f"  encrypt {i+1}/{N_TP}  eth_rx={nrx}  enc={ok/elapsed:.1f}/s  rx={nrx/elapsed:.1f}/s")

t_encrypt1 = time.time()
encrypt_s = t_encrypt1 - t_encrypt0
print(f"Encrypt loop done: {ok}/{N_TP} in {encrypt_s:.2f}s ({ok/encrypt_s:.1f} encrypts/s)")

target.fpga_write(REG_HS_RST, [1])
t_drain0 = time.time()
drain_deadline = t_drain0 + 120.0
while time.time() < drain_deadline:
    with eth_lock:
        nrx = eth_rx
    if nrx >= ok:
        break
    time.sleep(0.01)
else:
    raise TimeoutError(f"eth drain timeout: eth_rx={eth_rx}/{ok}")
t_drain1 = time.time()

total_s = t_drain1 - t_encrypt0
drain_s = t_drain1 - t_drain0
with eth_lock:
    nrx = eth_rx
    nser = len(eth_traces)

print("======== THROUGHPUT ========")
print(f"encrypts_ok     : {ok}/{N_TP}  (busy recoveries={stuck})")
print(f"eth_rx / serials: {nrx} / {nser}")
print(f"encrypt wall    : {encrypt_s:.2f}s  -> {ok/encrypt_s:.2f} encrypts/s")
print(f"drain wall      : {drain_s:.2f}s")
print(f"total wall      : {total_s:.2f}s  -> {nrx/total_s:.2f} traces/s (encrypt+drain)")
print(f"bits (rough)    : {nrx * 1024 * 16 / total_s / 1e6:.2f} Mbit/s payload @ 1024 x int16")
print("============================")
